In [1]:
!pip install ortools torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.

In [2]:
import os
import re
import numpy as np
import pandas as pd
import itertools
from sklearn.cluster import KMeans
from ortools.linear_solver import pywraplp

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv, global_mean_pool

# ==============================================================================
# DATASET LOADING & PREPROCESSING (Integrated User Logic)
# ==============================================================================
def load_all_datasets(folder_path):
    """
    Scans the given folder path and automatically loads all distance (C) 
    and demand (wij) matrices into dictionaries based on their node count.
    """
    distance_matrices = {}
    demand_matrices = {}
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None, None

    print(f"Scanning directory: {folder_path}...\n")
    
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        if not os.path.isfile(file_path):
            continue
            
        match = re.search(r'\d+', filename)
        if not match:
            continue
        nodes = int(match.group())
        
        try:
            # 1. Load Distance Matrices (C_ij)
            if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
                df = pd.read_csv(file_path, header=None)
                distance_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes. Shape: {df.values.shape}")
                
            # 2. Load Demand Matrices (W_ij)
            elif filename.startswith('wij'):
                if filename.endswith('.xlsx'):
                    df = pd.read_excel(file_path)
                elif filename.endswith('.csv'):
                    df = pd.read_csv(file_path)
                else:
                    continue
                
                # Ensure the matrix is exactly n x n by dropping extra index column if present
                if df.shape[1] > nodes:
                    df = df.iloc[:, 1:]
                    
                demand_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes. Shape: {df.values.shape}")
                
        except Exception as e:
            print(f"[X] Failed to load {filename}: {e}")

    return distance_matrices, demand_matrices

def calculate_node_production_attraction(W_matrix):
    """
    Calculates the production (O_i) and attraction (D_i) for each node.
    """
    production = np.sum(W_matrix, axis=1) 
    attraction = np.sum(W_matrix, axis=0) 
    return production, attraction

def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
    """
    Generates stochastic demand scenarios (W_ijs).
    Returns: A 3D NumPy array of shape (n, n, s)
    """
    np.random.seed(seed)
    n = W_matrix.shape[0]
    
    W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
    W_scenarios = np.maximum(W_scenarios, 0)
    W_scenarios = np.transpose(W_scenarios, (1, 2, 0))
    
    return W_scenarios

def calculate_path_costs(C_matrix, alpha=0.5):
    """
    Calculates the path cost parameter C_ijkm = C_ik + alpha * C_km + C_mj.
    Returns: A 4D NumPy array of shape (n, n, n, n) where axes are (i, j, k, m)
    """
    C_ik = C_matrix[:, np.newaxis, :, np.newaxis]
    C_km = C_matrix[np.newaxis, np.newaxis, :, :]
    C_mj = C_matrix.T[np.newaxis, :, np.newaxis, :]
    return C_ik + (alpha * C_km) + C_mj

# ==============================================================================
# DISCREPANCY 1: MATHEMATICAL MODEL (ROBUST SCENARIO-BASED SOLVER)
# ==============================================================================
def solve_exact_robust_model(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9, time_limit_mins=2):
    """
    Addresses the Screenshot and Docx: Implements the risk-averse, scenario-based 
    mathematical model using mu (threshold) and lambda_s (scenario violations).
    Now natively accepts the stochastic W_ijs array from generate_demand_scenarios.
    """
    N = len(distances)
    num_scenarios = W_scenarios.shape[2]
    
    # --- FIX: SCALE DATA TO PREVENT SCIP NUMERICAL CRASHES ---
    # Costs in the billions cause matrix inversion failures (Error <-6>) in SCIP.
    # We normalize the matrices to [0, 1] range during solving.
    dist_scale = np.max(distances) if np.max(distances) > 0 else 1.0
    w_scale = np.max(W_scenarios) if np.max(W_scenarios) > 0 else 1.0
    
    scaled_distances = distances / dist_scale
    scaled_W = W_scenarios / w_scale
    cost_multiplier = dist_scale * w_scale # To scale the final objective back up
    # ---------------------------------------------------------
    
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        raise Exception("SCIP solver not available.")

    # Set a time limit so the batch pipeline doesn't freeze on NP-Hard problems
    solver.SetTimeLimit(time_limit_mins * 60 * 1000)

    # 1. Decision Variables
    y = {} # y[k] = 1 if node k is a hub
    z = {} # z[i, k] = 1 if node i is allocated to hub k
    
    for k in range(N):
        y[k] = solver.IntVar(0, 1, f'y_{k}')
        for i in range(N):
            z[i, k] = solver.IntVar(0, 1, f'z_{i}_{k}')

    # Robust Variables (from the mathematical formulation)
    mu = solver.NumVar(0, solver.infinity(), 'mu')
    lambda_s = {s: solver.NumVar(0, solver.infinity(), f'lambda_{s}') for s in range(num_scenarios)}

    # 2. Constraints
    solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)

    for i in range(N):
        solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)

    for i in range(N):
        for k in range(N):
            solver.Add(z[i, k] <= y[k])

    # 3. Scenario Cost Calculation & Objective formulation
    for s in range(num_scenarios):
        # Extract the specific scenario matrix (n x n)
        W_s = scaled_W[:, :, s]
        
        cost_s_expr = 0
        for i in range(N):
            for k in range(N):
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[j, i] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * (alpha * scaled_distances[k, k]) * z[i, k]

        # Robust Constraint: lambda_s >= Cost_s - mu
        solver.Add(lambda_s[s] >= cost_s_expr - mu)

    # Objective: Minimize mu + (1 / (S * (1 - beta))) * Sum(lambda_s)
    penalty_weight = 1.0 / (num_scenarios * (1.0 - beta))
    solver.Minimize(mu + penalty_weight * solver.Sum([lambda_s[s] for s in range(num_scenarios)]))

    status = solver.Solve()

    # Accept FEASIBLE solutions because proving OPTIMALITY for N>=25 could take hours
    if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
        selected_hubs = [k for k in range(N) if y[k].solution_value() > 0.5]
        # Scale the objective value back up to its real-world magnitude
        real_cost = solver.Objective().Value() * cost_multiplier
        return selected_hubs, real_cost
    else:
        return None, float('inf')

# ==============================================================================
# DISCREPANCY 2: GRAPH REPRESENTATION (MULTI-GRAPH FOR DL)
# ==============================================================================
def create_multi_graph_data(distances, demands, labels=None):
    """
    Addresses Li et al. PDF: Constructs a PyTorch Geometric Data object with 
    MULTI-DIMENSIONAL edge attributes (Spatial, Demand, and Flow graphs combined).
    Now accepts 'labels' (y) for Neural Network training.
    """
    N = len(distances)
    
    # Use the user's calculation logic for Node Features
    production, attraction = calculate_node_production_attraction(demands)
    x = torch.tensor(np.column_stack((production, attraction)), dtype=torch.float)

    edge_index = []
    edge_attr = []

    for i in range(N):
        for j in range(N):
            if i != j:
                edge_index.append([i, j])
                dist = distances[i, j]
                dem = demands[i, j]
                flow = demands[i, j] + demands[j, i]
                edge_attr.append([dist, dem, flow])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    
    # Attach ground truth labels if provided (for training phase)
    if labels is not None:
        data.y = torch.tensor(labels, dtype=torch.float)
        
    return data

class MultiGraphHubRanker(torch.nn.Module):
    def __init__(self, node_in_dim=2, edge_in_dim=3, hidden_dim=64):
        super(MultiGraphHubRanker, self).__init__()
        self.conv1 = TransformerConv(node_in_dim, hidden_dim, edge_dim=edge_in_dim)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim, edge_dim=edge_in_dim)
        self.fc1 = torch.nn.Linear(hidden_dim, 32)
        self.fc2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x)).squeeze()

# ==============================================================================
# NEW: NEURAL NETWORK TRAINING PIPELINE
# ==============================================================================
def train_dl_model(model, train_data_list, epochs=50, lr=0.01):
    """
    Trains the MultiGraphHubRanker using the Exact Solver's output as Ground Truth.
    Uses Binary Cross Entropy Loss since we are predicting probability [0, 1].
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCELoss()
    
    model.train()
    print(f"\n[AI Training] Starting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        total_loss = 0
        for data in train_data_list:
            optimizer.zero_grad()
            
            # 1. Forward pass (predict probabilities)
            predictions = model(data)
            
            # 2. Calculate Loss against the Exact Solver labels
            loss = criterion(predictions, data.y)
            
            # 3. Backpropagation (update weights)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / len(train_data_list)
            print(f"   Epoch {epoch+1:03d}/{epochs} | Average Loss: {avg_loss:.4f}")
            
    print("[AI Training] Complete!")
    return model

# ==============================================================================
# DISCREPANCY 3: ALGORITHMIC PIPELINE (DL-CBS CLUSTERING & PRUNING)
# ==============================================================================
def run_dl_cbs(distances, demands, dl_rankings, p_hubs, C_ijkm=None, num_clusters=5, top_c_per_cluster=3):
    """
    Addresses Docx Step 9: Uses K-Means clustering and the AI rankings to drastically
    prune the search space, allowing it to scale effectively for N=100.
    """
    N = len(distances)
    
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(distances) 
    
    candidate_hubs = set()
    
    for cluster_idx in range(num_clusters):
        nodes_in_cluster = np.where(cluster_labels == cluster_idx)[0]
        cluster_rankings = [(node, dl_rankings[node]) for node in nodes_in_cluster]
        cluster_rankings.sort(key=lambda x: x[1], reverse=True)
        top_nodes = [node for node, prob in cluster_rankings[:top_c_per_cluster]]
        candidate_hubs.update(top_nodes)
        
    candidate_hubs = list(candidate_hubs)
    
    if len(candidate_hubs) < p_hubs:
        candidate_hubs = list(np.argsort(dl_rankings)[-p_hubs:])

    best_cost = float('inf')
    best_hubs = None
    
    for hub_combo in itertools.combinations(candidate_hubs, p_hubs):
        current_cost = evaluate_deterministic_cost(distances, demands, list(hub_combo), C_ijkm)
        if current_cost < best_cost:
            best_cost = current_cost
            best_hubs = list(hub_combo)
            
    return best_hubs, best_cost

def evaluate_deterministic_cost(distances, demands, hubs, C_ijkm=None, alpha=0.5):
    """Helper function to calculate cost given a set of hubs."""
    N = len(distances)
    cost = 0
    allocation = {}
    for i in range(N):
        allocation[i] = min(hubs, key=lambda h: distances[i, h])
        
    for i in range(N):
        for j in range(N):
            hi = allocation[i]
            hj = allocation[j]
            # Use precalculated vectorized path cost if available
            if C_ijkm is not None:
                path_cost = C_ijkm[i, j, hi, hj]
            else:
                path_cost = distances[i, hi] + alpha * distances[hi, hj] + distances[hj, j]
            cost += demands[i, j] * path_cost
            
    return cost

# ==============================================================================
# BATCH PIPELINE INTEGRATION (Phase 1, 2, and 3)
# ==============================================================================
if __name__ == "__main__":
    print("==================================================")
    print("🚀 STARTING FULL PIPELINE: DATA -> TRAIN -> EVAL")
    print("==================================================")
    
    # NOTE: Set this to your actual Kaggle Dataset path
    DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
    
    distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
    if distance_matrices and demand_matrices:
        common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
        
        training_graphs = []
        evaluation_data = []
        
        # ------------------------------------------------------------------
        # PHASE 1: LABEL GENERATION (Generate Ground Truth with Exact Solver)
        # ------------------------------------------------------------------
        print("\n\n" + "="*50)
        print("PHASE 1: LABEL GENERATION (Running Exact Solver)")
        print("="*50)
        for N in common_nodes:
            print(f"\nGenerating exact labels for Nodes = {N}...")
            distances = distance_matrices[N]
            demands = demand_matrices[N]
            
            W_scenarios = generate_demand_scenarios(demands, num_scenarios=100)
            C_ijkm = calculate_path_costs(distances, alpha=0.5)
            
            try:
                exact_hubs, exact_cost = solve_exact_robust_model(distances, W_scenarios, p_hubs=3, time_limit_mins=1)
                
                # Create Binary Labels array [0, 0, 1, 0, 1...] based on exact output
                labels = np.zeros(N)
                for h in exact_hubs:
                    labels[h] = 1.0
                
                # Build PyTorch graph with labels
                graph_data = create_multi_graph_data(distances, demands, labels=labels)
                training_graphs.append(graph_data)
                
                # Save raw data for phase 3 evaluation
                evaluation_data.append({
                    'N': N, 'distances': distances, 'demands': demands, 
                    'C_ijkm': C_ijkm, 'exact_cost': exact_cost, 'exact_hubs': exact_hubs,
                    'graph': graph_data
                })
                print(f"   [✓] Found Exact Hubs: {exact_hubs} | Cost: {exact_cost:.2f}")
                
            except Exception as e:
                print(f"   [!] Exact solve failed/timed out for N={N}: {e}")

        # ------------------------------------------------------------------
        # PHASE 2: TRAIN NEURAL NETWORK
        # ------------------------------------------------------------------
        print("\n\n" + "="*50)
        print("PHASE 2: TRAINING MULTI-GRAPH HUB RANKER")
        print("="*50)
        
        # Initialize the GNN model
        dl_model = MultiGraphHubRanker(node_in_dim=2, edge_in_dim=3, hidden_dim=64)
        
        if len(training_graphs) > 0:
            # Train the model using the graphs we just labeled
            dl_model = train_dl_model(dl_model, training_graphs, epochs=50, lr=0.01)
        else:
            print("[!] No training data generated. Skipping training.")

        # ------------------------------------------------------------------
        # PHASE 3: EVALUATION (AI-Driven DL-CBS)
        # ------------------------------------------------------------------
        print("\n\n" + "="*50)
        print("PHASE 3: INFERENCE & DL-CBS EVALUATION")
        print("="*50)
        
        dl_model.eval() # Set model to evaluation mode
        
        for data_dict in evaluation_data:
            N = data_dict['N']
            distances = data_dict['distances']
            demands = data_dict['demands']
            C_ijkm = data_dict['C_ijkm']
            graph_data = data_dict['graph']
            
            print(f"\nEvaluating DL-CBS for Nodes = {N}...")
            
            # --- THE MAGIC HAPPENS HERE ---
            # Instead of random numbers, ask the trained AI to rank the nodes!
            with torch.no_grad():
                ai_predicted_probs = dl_model(graph_data).numpy()
            
            # Run the heuristic using the AI's intelligent rankings
            dl_hubs, dl_cost = run_dl_cbs(
                distances, demands, ai_predicted_probs, 
                p_hubs=3, C_ijkm=C_ijkm, num_clusters=5, top_c_per_cluster=3
            )
            
            print(f"   Exact Cost:  {data_dict['exact_cost']:.2f} | Hubs: {data_dict['exact_hubs']}")
            print(f"   DL-CBS Cost: {dl_cost:.2f} | Hubs: {dl_hubs}")
            print(f"   Gap: {((dl_cost - data_dict['exact_cost']) / data_dict['exact_cost']) * 100:.2f}%")

    else:
        print("\n[!] Directory not found or empty. Please point DATA_FOLDER to your dataset path.")

🚀 STARTING FULL PIPELINE: DATA -> TRAIN -> EVAL
Scanning directory: /kaggle/input/datasets/infernalss/node-dataset...

[✓] Loaded Demand Matrix (W) for 100 nodes. Shape: (100, 100)
[✓] Loaded Distance Matrix (C) for 200 nodes. Shape: (200, 200)
[✓] Loaded Demand Matrix (W) for 200 nodes. Shape: (200, 200)
[✓] Loaded Demand Matrix (W) for 150 nodes. Shape: (150, 150)
[✓] Loaded Distance Matrix (C) for 150 nodes. Shape: (150, 150)
[✓] Loaded Distance Matrix (C) for 100 nodes. Shape: (100, 100)
[✓] Loaded Demand Matrix (W) for 25 nodes. Shape: (25, 25)
[✓] Loaded Distance Matrix (C) for 25 nodes. Shape: (25, 25)


PHASE 1: LABEL GENERATION (Running Exact Solver)

Generating exact labels for Nodes = 25...
   [✓] Found Exact Hubs: [3, 11, 16] | Cost: 5491206709.37

Generating exact labels for Nodes = 100...
   [✓] Found Exact Hubs: [10, 36, 69] | Cost: 55849.67

Generating exact labels for Nodes = 150...
   [✓] Found Exact Hubs: [11, 53, 99] | Cost: 56051.85

Generating exact labels for Nod